# RTT cross-check - client timing vs packets on the wire

**Purpose (Day 8):** prove the client's stopwatch is trustworthy. For classic UDP DNS the wire RTT (query packet -> answer packet in the pcap) is unambiguous, so the client RTT should be only slightly larger (Python/OS overhead).

Prepare the wire CSV first:
```
python capture/pcap_to_csv.py udp-rtt capture/pcaps/<file>.pcap --out results/<name>_udp_rtt.csv
```

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# find the repo root (the folder that contains client/) no matter where Jupyter was started
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "client").is_dir())
for sub in ("client", "ml", "capture"):
    sys.path.insert(0, str(ROOT / sub))
RESULTS = ROOT / "results"
print("repo root:", ROOT)

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
from compare_rtt import compare

# --- edit these two lines if you want specific files ---
WIRE_CSV = sorted(RESULTS.glob("*_udp_rtt.csv"))[-1]          # newest wire CSV
CLIENT_CSVS = sorted(glob.glob(str(RESULTS / "raw_*.csv")))    # all raw client files
print("wire  :", WIRE_CSV.name); print("client:", len(CLIENT_CSVS), "files")

m = compare(CLIENT_CSVS, str(WIRE_CSV))
print(f"matched {len(m)} classic-UDP queries")
m.gap_ms.describe().round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].scatter(m.wire_rtt_ms, m.client_rtt_ms, s=12); lim = [0, max(m.wire_rtt_ms.max(), m.client_rtt_ms.max()) * 1.05]
ax[0].plot(lim, lim, "k--", lw=1); ax[0].set_xlabel("wire RTT (ms)"); ax[0].set_ylabel("client RTT (ms)"); ax[0].set_title("should hug the diagonal")
ax[1].hist(m.gap_ms, bins=30); ax[1].set_xlabel("client - wire (ms)"); ax[1].set_title("gap")
fig.tight_layout(); plt.show()
print("VERDICT:", "OK - gap is small" if m.gap_ms.median() < 5 else "TOO LARGE - narrow the client's timer window")